> ⏸ **Parked: not part of the current phase.** This notebook will be updated once model training and evaluation (notebooks 01–03) are finalised. Don't run it yet.

# 04 · Grad-CAM++ localisation

Visualises where the classifier looks, and measures which facial regions it relies on for each manipulation method (share of Grad-CAM++ activation per region).

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 (not P100)
- **Internet:** On
- **Inputs:** output of **01_extract_faces**; output of **02_train**

**Produces:** `/kaggle/working/outputs/gradcam/` (examples + region attribution table). When it finishes, use **Save Version → Save & Run All**, then add this notebook's output as an input to the next notebook.

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
from src.kaggle_utils import setup
setup(data=True, checkpoint=True)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

from src.config import DATA_DIR, OUTPUT_DIR
from src.xai.gradcam import Explainer, draw_bbox

cam = Explainer()
m = pd.read_csv(DATA_DIR / "manifest.csv")
test = m[m.split == "test"]
path_of = lambda r: DATA_DIR / r.split / r.label / r.file
out = OUTPUT_DIR / "gradcam"; out.mkdir(parents=True, exist_ok=True)

## Examples per method
Box = connected hottest region. Title = prediction and the region with most activation.

In [ ]:
methods = sorted(test.method.unique())
fig, axes = plt.subplots(len(methods), 8, figsize=(17, 2.4 * len(methods)))
for row, method in zip(np.atleast_2d(axes), methods):
    for i, r in enumerate(test[test.method == method].sample(4, random_state=1).itertuples()):
        img = Image.open(path_of(r)); exp = cam(img)
        row[2 * i].imshow(img.resize((224, 224))); row[2 * i].set_title(method, fontsize=8)
        row[2 * i + 1].imshow(draw_bbox(exp.overlay, exp.bbox))
        row[2 * i + 1].set_title(f"{exp.label} {exp.prob_fake:.2f}\n{exp.top_regions[0][0]}", fontsize=8)
for ax in np.ravel(axes): ax.axis("off")
plt.tight_layout(); plt.savefig(out / "examples.png", dpi=120); plt.show()
print(exp.describe())

## Which regions does the model use for each manipulation?
Mean share of Grad-CAM++ activation per facial region, over `N` correctly classified test frames per method.

In [ ]:
N = 100
rows = []
for method in methods:
    for r in test[test.method == method].sample(min(N * 2, (test.method == method).sum()), random_state=0).itertuples():
        exp = cam(Image.open(path_of(r)))
        if (exp.label == "fake") == (r.label == "fake"):
            rows.append({"method": method, **exp.region_scores})
        if sum(x["method"] == method for x in rows) >= N:
            break
attr = pd.DataFrame(rows).groupby("method").mean()
attr.to_csv(out / "region_attribution.csv")

fig, ax = plt.subplots(figsize=(11, 0.6 * len(attr) + 1.5))
im = ax.imshow(attr.values, cmap="magma")
ax.set_xticks(range(attr.shape[1]), attr.columns, rotation=35, ha="right"); ax.set_yticks(range(len(attr)), attr.index)
for (i, j), v in np.ndenumerate(attr.values):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if v < attr.values.max() * 0.6 else "black", fontsize=8)
plt.colorbar(im, label="share of activation"); plt.title("Grad-CAM++ activation by facial region")
plt.tight_layout(); plt.savefig(out / "region_attribution.png", dpi=150); plt.show()